# Pilot
Twenty fixed questions for the selected track: plain model and that track's method. Results stay separate across tracks. Offline mode checks storage only and reports no accuracy. Live runs require the selected track's completed debugging run.

In [ ]:
import json
import os
from pathlib import Path
import pandas as pd
from tom_research.client import ModelConfig
from tom_research.evaluation import run_pilot, summarize
from tom_research.methods import LiveRunner, offline_runner
from tom_research.study import ROOT, load_study, prepare_pilot, provenance, track_paths

TRACK = os.environ.get('TOM_TRACK', 'thoughttracing')
LIVE = False
config = json.loads((ROOT / 'configs/pilot.json').read_text())
source, manifest_path = track_paths(TRACK)
prepare_pilot(source, manifest_path, track=TRACK)
debug, pilot, manifest = load_study(source, manifest_path, track=TRACK)
methods = manifest['methods']
run_config = {**config, 'track': TRACK, 'methods': methods,
              'answer_format': manifest['answer_format'], 'mode': 'live' if LIVE else 'offline',
              'sources': provenance(track=TRACK)}
result_root = Path(os.environ.get('TOM_RESULTS_DIR', str(ROOT / 'results' / run_config['mode']))) / TRACK
if LIVE:
    assert config['mode'] == 'live' and config['hardware'] and config['server_version'], 'Complete live preflight first'
    debug_manifest = json.loads((result_root / 'debug/manifest.json').read_text())
    assert debug_manifest['config'] == run_config, 'Debug settings differ from this run'
    debug_records = [json.loads(line) for line in (result_root / 'debug/predictions.jsonl').read_text().splitlines()]
    expected_debug = {(method, item.id) for method in methods for item in debug}
    assert {(record['method'], record['id']) for record in debug_records} == expected_debug
    assert all(record['status'] == 'ok' for record in debug_records), 'Resolve debugging errors first'
    runner = LiveRunner(ModelConfig(**config['model_config']), config['item_seconds'])
else:
    runner = offline_runner

In [ ]:
records = run_pilot(pilot, methods, runner, result_root / 'pilot', run_config)
print(f'{TRACK}: {len(records)} / {len(pilot) * len(methods)} records')
print('Mode:', run_config['mode'])
report = pd.DataFrame(summarize(records))
report

In [ ]:
errors = [record for record in records if record['status'] in ('error', 'invalid')
          or (record['status'] == 'ok' and not record['correct'])]
pd.DataFrame(errors[:5])

In [ ]:
print('Track:', TRACK, '| Dataset:', manifest['dataset'], '| Answers:', manifest['answer_format'])
if not LIVE:
    print('No research results yet. Synthetic records only exercise the pipeline.')

## Notes
Compare only the selected method and its plain baseline here. Report correct/total counts, invalid answers, transport errors, tokens and runtime. Errors remain in the denominator.
Do not compare these 20-question scores directly with published full-benchmark scores or rank methods across the two tracks. A later reproduction must match each paper's full evaluation subset and settings.
ThoughtTracing currently changes the action-labeling helper to Qwen. AutoToM's plain baseline uses exact-choice text formatting. Qwen thinking is disabled. Record these differences before interpreting model improvements.